In [1]:
# Criar o “banco local” (equivalente a SQLite)

import sqlite3, time, json, random, string, pathlib
from datetime import datetime

DB_PATH = "local_store.db"      # ↔ arquivo .db dentro do app
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS expense (
    id        INTEGER PRIMARY KEY AUTOINCREMENT,
    title     TEXT        NOT NULL,
    amount    REAL        NOT NULL,
    ts        INTEGER     NOT NULL,
    synced    INTEGER     NOT NULL DEFAULT 0
)
""")
conn.commit()

In [2]:
# Inserir registros “offline”

def add_expense(title, amt):
    cur.execute(
        "INSERT INTO expense(title, amount, ts, synced) VALUES (?, ?, ?, 0)",
        (title, amt, int(time.time()))
    )
    conn.commit()

add_expense("Café aeroporto",   14.50)
add_expense("Táxi hotel",       85.00)
add_expense("Almoço reunião",  129.90)

In [3]:
#Listar tudo que ainda não foi sincronizado

import pandas as pd

df_unsynced = pd.read_sql_query(
    "SELECT id, title, amount, datetime(ts, 'unixepoch') AS ts \
     FROM expense WHERE synced = 0",
    conn
)
df_unsynced

,id,title,amount,ts
0,1,Café aeroporto,14.5,2025-07-10 17:51:17
1,2,Táxi hotel,85.0,2025-07-10 17:51:17
2,3,Almoço reunião,129.9,2025-07-10 17:51:17


In [4]:
# Simular o WorkManager: gerar PATCH incremental

def build_patch(rows):
    # Converte linhas SQL → dicionário → JSON
    payload = [
        {"id": r["id"], "title": r["title"],
         "amount": r["amount"], "ts": r["ts"]}
        for _, r in rows.iterrows()
    ]
    return json.dumps(payload, indent=2)

patch_json = build_patch(df_unsynced)
print(patch_json)

[
  {
    "id": 1,
    "title": "Caf\u00e9 aeroporto",
    "amount": 14.5,
    "ts": "2025-07-10 17:51:17"
  },
  {
    "id": 2,
    "title": "T\u00e1xi hotel",
    "amount": 85.0,
    "ts": "2025-07-10 17:51:17"
  },
  {
    "id": 3,
    "title": "Almo\u00e7o reuni\u00e3o",
    "amount": 129.9,
    "ts": "2025-07-10 17:51:17"
  }
]


In [5]:
# “Servidor respondeu 200 OK” → marcar registros como sincronizados

ids = tuple(df_unsynced["id"].tolist())
cur.execute(f"UPDATE expense SET synced = 1 WHERE id IN {ids}")
conn.commit()

In [6]:
# Conferir que não há nada pendente

pd.read_sql_query("SELECT * FROM expense WHERE synced = 0", conn)

,id,title,amount,ts,synced
